<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Michi_v2/rule_based_matching_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Rule-based matching (baseline)**

In [ ]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

Cloning into 'DataScienceCapstoneProject'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 232 (delta 98), reused 20 (delta 20), pack-reused 88 (from 1)
Receiving objects: 100% (232/232), 1.71 MiB | 14.09 MiB/s, done.
Resolving deltas: 100% (131/131), done.


In [ ]:
%cd DataScienceCapstoneProject
!git checkout Michi_v2

/content/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject
Branch 'Michi_v2' set up to track remote branch 'Michi_v2' from 'origin'.
Switched to a new branch 'Michi_v2'


In [ ]:
!ls

department.csv			       linkedin-cvs-not-annotated.csv
df_profiles_cleansed.csv	       linkedin-cvs-not-annotated.json
FeatureEngineering+RandomForest.ipynb  README.md
linkedin-cvs-annotated.json	       seniority.csv
linkedin-cvs-annotatedV5.csv


**Task:  Identify relevant job titles and text passages using predefined label lists and assign domain and seniority accordingly.**

The rule-based matching model is a pure exact matching lookup. This means that in the first step, we will not insert any of our own rules, etc.

In [ ]:
import pandas as pd

seniority_df = pd.read_csv("seniority.csv")
department_df = pd.read_csv("department.csv")

In [ ]:
seniority_df.head(10)

,text,label
0,Analyst,Junior
1,Analyste financier,Junior
2,Anwendungstechnischer Mitarbeiter,Junior
3,Application Engineer,Senior
4,Applications Engineer,Senior
5,Architecte SI - Chef de projet Applicatif,Lead
6,Associate,Junior
7,Associate - Research,Junior
8,Associate Partner,Junior
9,Associate Recruiter,Junior


In [ ]:
department_df.head(10)

,text,label
0,Adjoint directeur communication,Marketing
1,Advisor Strategy and Projects,Project Management
2,Beratung & Projekte,Project Management
3,Beratung & Projektmanagement,Project Management
4,Beratung und Projektmanagement kommunale Partner,Project Management
5,Cadre marketing digital,Marketing
6,Chargé de communication,Marketing
7,Chargé de communication digitale,Marketing
8,Chargé de communication et marketing,Marketing
9,Chargé de Webmarketing SEO/SEA,Marketing


We apply basic text normalization (lowercasing, trimming, whitespace collapsing, and german character transliteration such as ä→ae, ö→oe, ü→ue, ß→ss) consistently across all datasets and perform exact matching on the normalized job titles.

In [ ]:
import re

GERMAN_MAP = str.maketrans({
    "ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss",
    "Ä": "ae", "Ö": "oe", "Ü": "ue"
})

def normalize_title(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = s.translate(GERMAN_MAP)
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    return s

seniority_df["text_norm"] = seniority_df["text"].apply(normalize_title)
department_df["text_norm"] = department_df["text"].apply(normalize_title)

In both Datasets we have a list of job titles and the correct label. For our rule based matching we just apply exact matching. For this we create dictionaries for the lookup.

In [ ]:
seniority_lookup = dict(zip(seniority_df["text_norm"], seniority_df["label"]))
department_lookup = dict(zip(department_df["text_norm"], department_df["label"]))

Baseline inference pipeline based on exact string matching:

In [ ]:
def baseline_predict(title):
    seniority = seniority_lookup.get(title, "unknown")
    department = department_lookup.get(title, "unknown")
    return seniority, department

The pipeline takes a job title as input and performs an exact string match. If the job title is found exactly, the corresponding labels are applied; otherwise, the label is set to unknown.

**Evaluation**

For evaluation, we use the test data JSON file. From the previous steps, we have already converted the JSON file into a CSV file.

In [ ]:
eval_df = pd.read_csv("df_profiles_cleansed.csv")
eval_df.head(10)

,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0,6,6.339726
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0,6,6.424658
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
5,Nagel Car Group,Buchhalterin,2000-05,2019-06,INACTIVE,Other,Professional,0,6,19.095890
6,Computer Solutions,Solutions Architect,2024-03,2025-12,ACTIVE,Information Technology,Professional,1,8,1.753425
7,Computer Solutions,Senior Network Engineer,2019-07,2024-03,INACTIVE,Information Technology,Senior,1,8,4.671233
8,Texas A&M University-Corpus Christi,Manager of Network Services,2017-02,2019-07,INACTIVE,Information Technology,Professional,1,8,2.410959
9,Texas A&M University-Corpus Christi,Infrastructure Administrator II,2015-06,2017-02,INACTIVE,Information Technology,Professional,1,8,1.673973


In [ ]:
#We want to use only the ACTIVE positions of our test data set:
eval_active_df = eval_df[eval_df["status"] == "ACTIVE"].copy()

In [ ]:
#text normalization eval test set
eval_active_df["position_norm"] = eval_active_df["position"].apply(normalize_title)

In [ ]:
#baseline prediction (exact match on normalized keys)
eval_active_df[["pred_seniority", "pred_department"]] = eval_active_df["position_norm"].apply(
    lambda t: pd.Series(baseline_predict(t))
)

Metrics und Analysis:

In [ ]:
#coverage of exact string matching
coverage_sen = (eval_active_df["pred_seniority"] != "unknown").mean()
coverage_dep = (eval_active_df["pred_department"] != "unknown").mean()

print("Coverage seniority:", round(coverage_sen, 4))
print("Coverage department:", round(coverage_dep, 4))

Coverage seniority: 0.2187
Coverage department: 0.0627


The coverage of around 22% shows that only a small proportion of active job titles in the evaluation data set are exactly included in the seniority label list. This highlights the severe limitations of a pure lookup approach when dealing with varied, real-world job titles.
With only 6% coverage, the department baseline is particularly restrictive. This suggests that department names are likely formulated in a much more heterogeneous manner in practice than seniority titles and therefore cannot be matched exactly.

In [ ]:
#seniority accuracy on only matched titles
mask_sen = (eval_active_df["pred_seniority"] != "unknown") & eval_active_df["seniority"].notna()
acc_sen_matched = (eval_active_df.loc[mask_sen, "pred_seniority"] == eval_active_df.loc[mask_sen, "seniority"]).mean()

print("Seniority accuracy (matched only):", round(acc_sen_matched, 4))
print("Matched seniority:", int(mask_sen.sum()))

Seniority accuracy (matched only): 0.707
Matched seniority: 157


For matched cases, the baseline achieves an accuracy of 70.7%, which shows that the assignment is predominantly correct where an exact match exists. Errors are likely to arise from ambiguous titles or inconsistent seniority definitions.

In [ ]:
#department accuracy on only matched titles
mask_dep = (eval_active_df["pred_department"] != "unknown") & eval_active_df["department"].notna()
acc_dep_matched = (eval_active_df.loc[mask_dep, "pred_department"] == eval_active_df.loc[mask_dep, "department"]).mean()

print("Department accuracy (matched only):", round(acc_dep_matched, 4))
print("Matched department:", int(mask_dep.sum()))

Department accuracy (matched only): 0.9333
Matched department: 45


The very high accuracy of 93.3% on matched department titles shows that precisely listed department names are almost unique. Once a match exists, the classification is therefore very reliable.

In [ ]:
#applying end to end accuracy
acc_sen_e2e = (
    eval_active_df["seniority"].notna() &
    (eval_active_df["pred_seniority"] == eval_active_df["seniority"])
).mean()

acc_dep_e2e = (
    eval_active_df["department"].notna() &
    (eval_active_df["pred_department"] == eval_active_df["department"])
).mean()

print("Seniority end-to-end accuracy:", round(acc_sen_e2e, 4))
print("Department end-to-end accuracy:", round(acc_dep_e2e, 4))

Seniority end-to-end accuracy: 0.1546
Department end-to-end accuracy: 0.0585


As expected, the end-to-end accuracy of 15.5% is low, as all unknown predictions count as errors. This illustrates that the main weakness of the baseline is not incorrect assignments, but rather a lack of coverage.

At 5.9%, the end-to-end accuracy for department is even lower. This is a direct consequence of the extremely low coverage and confirms the limited practicality of pure exact matching for department predictions.

In [ ]:
from sklearn.metrics import classification_report

#classification report for matched seniorities
print("Seniority classification report (matched only):")
print(classification_report(
    eval_active_df.loc[mask_sen, "seniority"],
    eval_active_df.loc[mask_sen, "pred_seniority"],
    zero_division=0
))

#classificiation report for matched departments
print("Department classification report (matched only):")
print(classification_report(
    eval_active_df.loc[mask_dep, "department"],
    eval_active_df.loc[mask_dep, "pred_department"],
    zero_division=0
))

Seniority classification report (matched only):
              precision    recall  f1-score   support

    Director       0.40      1.00      0.57        10
      Junior       0.33      0.50      0.40         2
        Lead       0.94      0.89      0.91        18
  Management       1.00      0.77      0.87        97
Professional       0.00      0.00      0.00        21
      Senior       0.24      1.00      0.39         9

    accuracy                           0.71       157
   macro avg       0.49      0.69      0.52       157
weighted avg       0.77      0.71      0.71       157

Department classification report (matched only):
                        precision    recall  f1-score   support

        Administrative       1.00      1.00      1.00         1
  Business Development       0.75      1.00      0.86         3
            Consulting       1.00      0.71      0.83         7
Information Technology       0.67      1.00      0.80         2
             Marketing       0.50      

The ‘Professional’ class is not included in the list of seniority designations used for the baseline search. Consequently, the baseline cannot predict this class, resulting in zero precision and recall for ‘Professional’ in the evaluation set. This highlights a discrepancy between the predefined designation lists and the annotated evaluation data. Apart from that, the high recall rate for ‘Director’ and ‘Senior’ combined with low precision indicates a tendency to overclassify similar titles.

Most department classes show very high precision and recall values, especially “Sales”, ‘Project Management’ and ‘Purchasing’. Errors mainly occur in smaller classes.

In [ ]:
from sklearn.metrics import confusion_matrix

cm_sen = confusion_matrix(
    eval_active_df.loc[mask_sen, "seniority"],
    eval_active_df.loc[mask_sen, "pred_seniority"],
)

labels_sen = sorted(eval_active_df.loc[mask_sen, "seniority"].unique())
cm_sen_df = pd.DataFrame(cm_sen, index=labels_sen, columns=labels_sen)
cm_sen_df

,Director,Junior,Lead,Management,Professional,Senior
Director,10,0,0,0,0,0
Junior,0,1,0,0,0,1
Lead,0,0,16,0,0,2
Management,15,0,1,75,0,6
Professional,0,2,0,0,0,19
Senior,0,0,0,0,0,9


The confusion matrix shows that management titles are sometimes misclassified as director or senior, which indicates semantic proximity between these roles. Professional is often interpreted as senior or junior, which reflects the conceptual vagueness of this category.

In [ ]:
cm_dep = confusion_matrix(
    eval_active_df.loc[mask_dep, "department"],
    eval_active_df.loc[mask_dep, "pred_department"],
)

labels_dep = sorted(eval_active_df.loc[mask_dep, "department"].unique())
cm_dep_df = pd.DataFrame(cm_dep, index=labels_dep, columns=labels_dep)
cm_dep_df

,Administrative,Business Development,Consulting,Information Technology,Marketing,Project Management,Purchasing,Sales
Administrative,1,0,0,0,0,0,0,0
Business Development,0,3,0,0,0,0,0,0
Consulting,0,1,5,1,0,0,0,0
Information Technology,0,0,0,2,0,0,0,0
Marketing,0,0,0,0,1,0,0,0
Project Management,0,0,0,0,0,10,0,0
Purchasing,0,0,0,0,0,0,2,0
Sales,0,0,0,0,1,0,0,18


The confusion matrix for department shows that only very few isolated errors occur during classification.

**Summary:**

The exact matching baseline shows very limited coverage on real-world, annotated LinkedIn CV data, especially for department predictions. At the same time, classification accuracy is high on matched cases, confirming that the underlying label lists are correct and consistent. The low end-to-end accuracy results primarily from the high number of unknown predictions and not from systematically incorrect assignments. Overall, the baseline serves as a lower performance limit, which clearly illustrates that simple lookup methods do not scale sufficiently for heterogeneous job titles and motivates the use of advanced rule-based or learning-based approaches.